# Agile effort estimation improved by feature selection and model explainability

## 1. Dataset aggregation
We download the datasets published in `https://github.com/morakotch/datasets` in `/datasets`. Each project dataset has a separate csv for project issues and another for iteration data. In order to add statistical aggregations of the features of issues into each iteration, as described in [1], we run a script that creates a new csv after merging both issue and iteration csv files. For this study, and in accordance with the cited work, we will be using the datasets at the 30% of their estimated duration (those ended in `_30`).

- [1] Choetkiertikul, M., Dam, H. K., Tran, T., Ghose, A., & Grundy, J. (2018). Predicting Delivery Capability in Iterative Software Development. IEEE Transactions on Software Engineering, 44(6), 551–573. https://doi.org/10.1109/TSE.2017.2693989

In [1]:
%run src/features_aggregator.py --iterations "datasets/Apache/apache_iteration_30.csv" --issues "datasets/Apache/apache_issue_30.csv" --output_iterations "datasets/apache_iteration_30_features.csv"
%run src/features_aggregator.py --iterations "datasets/JBoss/jboss_iteration_30.csv" --issues "datasets/JBoss/jboss_issue_30.csv" --output_iterations "datasets/jboss_iteration_30_features.csv"
%run src/features_aggregator.py --iterations "datasets/JIRA/jira_iteration_30.csv" --issues "datasets/JIRA/jira_issue_30.csv" --output_iterations "datasets/jira_iteration_30_features.csv"
%run src/features_aggregator.py --iterations "datasets/MongoDB/mongodb_iteration_30.csv" --issues "datasets/MongoDB/mongodb_issue_30.csv" --output_iterations "datasets/mongodb_iteration_30_features.csv"
%run src/features_aggregator.py --iterations "datasets/Spring/spring_iteration_30.csv" --issues "datasets/Spring/spring_issue_30.csv" --output_iterations "datasets/spring_iteration_30_features.csv"

Reading iterations from: datasets/Apache/apache_iteration_30.csv and issues from: datasets/Apache/apache_issue_30.csv
Nº Iterations: 348
Nº Issues: 5826
Aggregating features. Progress:[###################################]
Features were successfully aggregated and stored in: datasets/apache_iteration_30_features.csv
Reading iterations from: datasets/JBoss/jboss_iteration_30.csv and issues from: datasets/JBoss/jboss_issue_30.csv
Nº Iterations: 372
Nº Issues: 4984
Aggregating features. Progress:[######################################]
Features were successfully aggregated and stored in: datasets/jboss_iteration_30_features.csv
Reading iterations from: datasets/JIRA/jira_iteration_30.csv and issues from: datasets/JIRA/jira_issue_30.csv
Nº Iterations: 1873
Nº Issues: 10852
Aggregating features. Progress:[############################################################################################################################################################################################]

## 2. Feature Subset Selection
We aim to reduce the set of features used to train a machine learning model, a Random Forest (RF) in this work. We use two Feature Subset Selection (FSS) methods, described in the following sections. The RF model and the cross validation folding strategy are defined below.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from src.custom_kfold import CustomKFold
from src.train_dataset import get_X_y

# sample project:
dataset = "apache_iteration_30_features"

dataset_iteration_30 = pd.read_csv(
    f"datasets/{dataset}.csv")
X, y = get_X_y(dataset_iteration_30)

# iterations of the dataset are sorted chronologically by their start date, and every ith iteration out of ten is included in the ith fold.
cv = CustomKFold(n_splits=10, shuffle=False)

random_forest = RandomForestRegressor(
    n_estimators=500, max_depth=7, random_state=42
)

### 2.1 FSS: Filter with SelectPercentile
This method selects a percentile of features that contribute most to a given scoring function in the model. Specifically, our filter method selects the nth percentile of features based on their importances, obtained by training the RF model in advance.

In [3]:
import json
from src.get_folds_results import get_fold_results
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectPercentile


def custom_scorer(X, y):
    random_forest.fit(X, y)
    return random_forest.feature_importances_


percentile = 20  # select only 20% most important features

sfs_pipeline = Pipeline(
    [
        (
            "feature_selection",
            SelectPercentile(percentile=percentile,
                             score_func=custom_scorer),
        ),
        ("regressor", random_forest),
    ]
)

execution_result = get_fold_results(
    sfs_pipeline,
    cv=cv,
    X=X,
    y=y,
    config_text=f"SelectPercentile {percentile}%",
    dataset=dataset
)
print(json.dumps(execution_result))

{"config": "SelectPercentile 20%", "dataset": "apache_iteration_30_features", "mae_avg": 4.732863077565051, "nmae_avg": 0.33001028439571656, "time(s)": 44.075655698776245, "folds": [{"mae": 4.866305194717398, "nmae": 0.270350288595411, "features": ["no_issue_starttime", "vel_starttime", "vel_removed", "vel_todo", "vel_inprogress", "no_teammember", "no_issues", "type_freq_Improvement", "priority_freq_Major", "no_fixversion_std", "no_fixversion_var", "no_des_change_mean", "no_des_change_std", "no_des_change_var", "gunning_fog_freq_easy", "type_freq_Bug", "gunning_fog_freq_hard", "type_freq_Epic", "type_freq_Story", "type_freq_Documentation"]}, {"mae": 5.164624234883973, "nmae": 0.26485252486584476, "features": ["vel_starttime", "no_issue_removed", "vel_removed", "vel_todo", "vel_inprogress", "no_teammember", "no_issues", "type_freq_Improvement", "priority_freq_Major", "priority_freq_Minor", "no_fixversion_mean", "no_des_change_mean", "no_des_change_std", "no_des_change_var", "gunning_fog

### 2.2 FSS: Wrapper with SequentialFeatureSelector
This method adds features in a greedy manner to form a feature subset. At each stage, it selects the best feature to add based on the RF model’s cross-validation score, which is the Mean Absolute Error (MAE). We configured this method to select features until a threshold tolerance of 0.001 does not change between two consecutive feature additions.

In [ ]:

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import make_scorer

from src.sdee_statistics import calculate_mae

mae_scorer = make_scorer(calculate_mae, greater_is_better=False)

sfs_pipeline = Pipeline(
        [
            (
                "feature_selection",
                SequentialFeatureSelector(
                    random_forest,
                    direction="forward",
                    n_features_to_select="auto",
                    tol=0.001,
                    cv=CustomKFold(n_splits=2, shuffle=False),
                    n_jobs=-1,
                    scoring=mae_scorer,
                ),
            ),
            ("regressor", random_forest),
        ]
    )

execution_result = get_fold_results(
    sfs_pipeline,
    cv=cv,
    X=X,
    y=y,
    config_text=f"tol=0.001 direction=forward n_features_to_select=auto",
    dataset=dataset
)
print(json.dumps(execution_result))

## 3. Run the experiment
We run a experiment to find the minimum subset of features to use without obtaining a statistically significant impact in predicting performance. We use a Slurm queue to run the batch jobs. 

- For the `SelectPercentile`, we run jobs for the percentiles `[1, 5, 10, 20, 40, 60, 80, 100]`. 
- For the `SequentialFeatureSelector`, we run a forward auto selection with the tolerance defined above.

In [ ]:
!bash experiment_select_percentile.sh
!bash experiment_sequential.sh

To merge all config results into a single json file, we use the following script: `python results_merger.py <slurm_job_group_folder>`, that creates a `outputs/merged_output.json` file with all the results of all configurations. In our environment, each job is storing a separate json file under `/<slurm_job_group_folder>/outputs`.

## 4. Run statistical tests
Since we are comparing multiple models, we first use the non-parametric Friedman test to assess whether there are significant differences across the groups of data for each project separately. 

If significant differences are found, we then use a paired Wilcoxon signed rank test to compare each feature selection configuration against the full set of features. We apply the two-sided version of the Wilcoxon test because we are interested in detecting any significant differences in performance—whether positive or negative—between the reduced feature sets and the full feature set. For this analysis, we set a significance level of α = 0.05. 

We also applied Holm-Bonferroni correction to address the possibility of obtaining false negatives after multiple comparisons. Additionally, it is of interest to quantify the effect size of the two methods being compared. For that purpose, and following recommendations from the literature, we also apply the non-parametric Vargha and Delaney’s ˆA12 statistic.

In [5]:
%run statistical_tests.py "outputs/merged_output.json"

Dataset: apache_iteration_30_features
Friedman Test - Statistic: 40.10666666666668, p-value: 0.0000
Statistically significant differences found. Proceeding with pairwise Wilcoxon tests.
Config: SelectPercentile 100%, Mean MAE: 4.7651
Config: SelectPercentile 80%, Mean MAE: 4.7691 (p=1.0000, A12=0.520)
Config: SelectPercentile 60%, Mean MAE: 4.7642 (p=0.8457, A12=0.470)
Config: SelectPercentile 40%, Mean MAE: 4.7383 (p=0.3750, A12=0.510)
Config: SelectPercentile 20%, Mean MAE: 4.7546 (p=0.7695, A12=0.520)
Config: SelectPercentile 10%, Mean MAE: 5.0205 (p=0.0371, A12=0.620)
Config: SelectPercentile 5%, Mean MAE: 5.2476 (p=0.0488, A12=0.660)
Config: SelectPercentile 1%, Mean MAE: 7.2539 (p=0.0020, A12=0.960)
Config: tol=0.001 direction=forward n_features_to_select=auto, Mean MAE: 5.2808 (p=0.0273, A12=0.640)

Holm-Bonferroni Adjusted p-values:
Config: SelectPercentile 80%, Adjusted p-value: 1.0000
Config: SelectPercentile 60%, Adjusted p-value: 1.0000
Config: SelectPercentile 40%, Adjuste

## 5. Explainability with SHAP
We will analyze the most important features using the model with the minimum subset of features, obtained from the statistical tests. We will use the SHAP library to generate plots of the top 5 most important features and how the influence predictions.

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import shap
from sklearn.model_selection import train_test_split

feature_title_mapping = {
    'vel_todo': 'To-do Velocity',
    'vel_starttime': 'Velocity at Start Time',
    'gunning_fog_freq_hard': 'Frequency of Gunning Fog Hard',
    'type_freq_Improvement': 'Frequency of Type Improvement',
    'priority_freq_Major': 'Frequency of Priority Major',
    'no_issue_starttime': 'No. of Issues at Start Time',
    'vel_inprogress': 'In-Progress Velocity',
    'no_des_change_mean': 'Mean No. of Description Changes',
    'no_issuetodo': "No. of To-Do Issues",
    'priority_freq_Major - P3': "Frequency of Priority Major-P3",

}


datasets = [
    "apache_iteration_30_features",
    "jboss_iteration_30_features",
    "jira_iteration_30_features",
    "mongodb_iteration_30_features",
    "spring_iteration_30_features",
]

for dataset in datasets:
    print("================================")
    print(f"Dataset {dataset}")
    dataset_iteration_30 = pd.read_csv(f"datasets/{dataset}.csv")
    # dataset_iteration_30.rename(columns=feature_title_mapping, inplace=True)
    dataset_iteration_30.head(1)

    # Setup model:
    X, y = get_X_y(dataset_iteration_30)
    num_features_original = X.shape[1]
    print(f"Number of features in original dataset: {num_features_original}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=0
    )

    # Define model
    random_forest = RandomForestRegressor(
        n_estimators=500, max_depth=7, random_state=42
    )

    def custom_scorer(X, y):
        random_forest.fit(X, y)
        return random_forest.feature_importances_

    # Create pipeline with feature selection
    sfs_pipeline = Pipeline(
        [
            (
                "feature_selection",
                SelectPercentile(percentile=10, score_func=custom_scorer),
            ),
            ("regressor", random_forest),
        ]
    )

    X_reduced = sfs_pipeline.named_steps["feature_selection"].fit_transform(
        X_train, y_train)

    selected_features_mask = sfs_pipeline.named_steps["feature_selection"].get_support(
    )
    selected_feature_names = X_train.columns[selected_features_mask]
    X_reduced = pd.DataFrame(X_reduced, columns=selected_feature_names)

    num_features_selected = selected_features_mask.sum()
    print(
        f"Number of features selected by SelectPercentile: {num_features_selected}")

    sfs_pipeline.named_steps["regressor"].fit(X_reduced, y_train)

    # Generate SHAP values for the selected features
    explainer = shap.TreeExplainer(sfs_pipeline.named_steps["regressor"])
    shap_values = explainer(X_reduced)

    # ranking of mean absolute SHAP values
    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
    feature_importance = pd.DataFrame({
        'feature': selected_feature_names,
        'mean_abs_shap': mean_abs_shap
    })
    feature_importance = feature_importance.sort_values(
        by='mean_abs_shap', ascending=False).reset_index(drop=True)
    for idx, row in feature_importance.head(10).iterrows():
        print(row.feature)

    # Map the feature names for SHAP beeswarm plot
    # mapped_feature_names = [feature_title_mapping.get(f, f) for f in selected_feature_names]

    # Generate SHAP beeswarm plot using only the selected features
    shap.plots.beeswarm(shap_values, max_display=6)

    # Save the plot
    plt.savefig(
        f'outputs/shap/test_{dataset}_beeswarm.png', bbox_inches='tight')
    plt.close()

Dataset apache_iteration_30_features
Number of features in original dataset: 97
Number of features selected by SelectPercentile: 10
vel_todo
vel_starttime
gunning_fog_freq_hard
type_freq_Improvement
priority_freq_Major
vel_inprogress
no_fixversion_mean
type_freq_Documentation
no_issue_starttime
no_teammember
Dataset jboss_iteration_30_features
Number of features in original dataset: 96
Number of features selected by SelectPercentile: 10
vel_todo
no_issue_starttime
vel_starttime
vel_inprogress
no_des_change_mean
no_issuetodo
no_fixversion_change_mean
no_issueinprogress
gunning_fog_freq_easy
no_fixversion_change_var
Dataset jira_iteration_30_features
Number of features in original dataset: 104
Number of features selected by SelectPercentile: 11
vel_todo
type_freq_Improvement
priority_freq_Major
vel_starttime
vel_inprogress
type_freq_Bug
priority_freq_Critical
gunning_fog_freq_hard
no_issues
no_issue_removed
Dataset mongodb_iteration_30_features
Number of features in original dataset: 96
